# Basic single-fish calcium analysis
This notebook configures and displays the reusable five-plane analysis implemented in `analysis/`.

## Cell 01 — Import analysis tools
Import the reusable processing and plotting functions from the `analysis` package.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from analysis.plane_merge import merge_planes
from analysis.plotting import (
    build_stimulus_colors,
    draw_population_mean,
    draw_raster,
    draw_stimulus_markers,
    draw_vertical_lines,
    robust_color_limit,
    save_figure_safely,
    stimulus_legend_handles,
)
from analysis.stimulus_analysis import (
    build_trial_aligned_traces,
    compute_response_metrics,
    concatenate_stimulus_mean_rasters,
    correlation_sort_order,
    load_experiment_stimuli,
    movement_onset_times,
    summarize_metric,
)

## Cell 02 — Configure the experiment
Set the experiment paths, acquisition parameters, plane selection, trial window, stimulus order, and plot-saving options.

In [ ]:
EXPERIMENT_DIR = Path(r"C:\Users\suribear\OneDrive - Université de Lausanne\Lab\Data\2p\L588_f01")
FISH_ID = "L588_f01"
STIMULI_DIR = Path(r"C:\Users\suribear\OneDrive - Université de Lausanne\Lab\Data\stimuli\Exp_3_rocking_2")
SELECTED_BLOCKS = ["B1", "B2", "B3", "B4"]
IMAGING_FPS = 2.0
STIMULUS_FPS = 60.0
PLANE_INDICES = "all"  # Or use an explicit list such as [0, 1, 2, 3, 4] or [1, 2]
PRE_STIMULUS_SEC = 5.0
POST_STIMULUS_MARGIN_SEC = 10.0  # Window = pre + each stimulus's own duration + this margin
SORT_BY_CORRELATION = True
STIMULUS_ORDER = None  # Or use exact names, for example ["DRiF", "Ri2F"]
SAVE_PLOTS = True
PLOT_DPI = 300
PLOTS_DIR = EXPERIMENT_DIR / "04_plots"

## Cell 03 — Plot settings
Colormap, colour limits, stimulus colours, and figure sizes shared by every figure below.

In [ ]:
RASTER_CMAP = "gray_r"
RASTER_VMIN = 0.0
RASTER_VMAX = None  # None: use the RASTER_PERCENTILE of each raster's values
RASTER_PERCENTILE = 99.0
STIMULUS_COLORMAP = "tab20"  # Colours for stimuli not listed in STIMULUS_COLORS
STIMULUS_COLORS = {}  # Optional fixed colours, for example {"LeB": "tab:red", "RiB": "tab:blue"}
ACTIVITY_FIGURE_SIZE = (15, 7)
ACTIVITY_WIDTH_RATIOS = [1.0, 0.035, 0.18]  # Raster and trace, colorbar, legend
ACTIVITY_HEIGHT_RATIOS = [4, 1]  # Raster, population mean
METRIC_FIGURE_SIZE = (10, 4.5)

## Cell 04 — Merge the selected imaging planes
Load each selected plane's dF/F and join them in memory; `roi_mapping` links every column back to its plane and Suite2P ROI.

In [ ]:
merged = merge_planes(EXPERIMENT_DIR, FISH_ID, plane_indices=PLANE_INDICES)
dfof = merged["dfof"]
roi_mapping = merged["mapping"]
neuron_order = correlation_sort_order(dfof) if SORT_BY_CORRELATION else np.arange(dfof.shape[1])
# Raster row i shows the neuron described in sorted_roi_mapping row i (plane and Suite2P ROI)
sorted_roi_mapping = roi_mapping.iloc[neuron_order].reset_index(drop=True)
print(f"Merged dF/F shape (time × neurons): {dfof.shape}")
print(roi_mapping.groupby("plane").size().rename("neurons"))

## Cell 05 — Load the stimulus timeline
Match selected block-log events and trajectory durations to the merged imaging frames, and assign one colour per stimulus.

In [ ]:
stimuli = load_experiment_stimuli(
    experiment_dir=EXPERIMENT_DIR,
    stimuli_dir=STIMULI_DIR,
    selected_blocks=SELECTED_BLOCKS,
    n_imaging_frames=dfof.shape[0],
    imaging_fps=IMAGING_FPS,
    stimulus_fps=STIMULUS_FPS,
)
stimulus_names = list(stimuli["stimulus_id_map"])
stimulus_colors = build_stimulus_colors(stimulus_names, STIMULUS_COLORMAP, STIMULUS_COLORS)
print(stimuli["stimulus_events"].groupby("stimulus_name").size().rename("trials"))

## Cell 06 — Plot the complete experiment
Merged neuron raster and population-average trace, with stimulus movement onsets.

In [ ]:
time_sec = np.arange(dfof.shape[0]) / IMAGING_FPS
recording_limits = (time_sec[0], time_sec[-1])
full_raster = dfof[:, neuron_order].T
full_vmax = RASTER_VMAX if RASTER_VMAX is not None else robust_color_limit([dfof], RASTER_PERCENTILE, RASTER_VMIN)
movement_onsets_sec = movement_onset_times(stimuli["stimulus_events"], stimuli["stimulus_durations"])
event_names = stimuli["stimulus_events"]["stimulus_name"].tolist()

full_figure = plt.figure(figsize=ACTIVITY_FIGURE_SIZE, constrained_layout=True)
full_grid = full_figure.add_gridspec(2, 3, width_ratios=ACTIVITY_WIDTH_RATIOS, height_ratios=ACTIVITY_HEIGHT_RATIOS)
full_raster_axis = full_figure.add_subplot(full_grid[0, 0])
full_mean_axis = full_figure.add_subplot(full_grid[1, 0], sharex=full_raster_axis)
full_colorbar_axis = full_figure.add_subplot(full_grid[0, 1])
full_legend_axis = full_figure.add_subplot(full_grid[0, 2])

draw_raster(full_raster_axis, full_raster, *recording_limits, RASTER_CMAP, RASTER_VMIN, full_vmax)
draw_population_mean(full_mean_axis, full_raster, time_sec, color="black")
for axis in (full_raster_axis, full_mean_axis):
    draw_stimulus_markers(axis, movement_onsets_sec, event_names, stimulus_colors, recording_limits, alpha=0.75)
    axis.spines[["top", "right"]].set_visible(False)

full_figure.colorbar(full_raster_axis.images[-1], cax=full_colorbar_axis, label="dF/F")
full_legend_axis.axis("off")
full_legend_axis.legend(handles=stimulus_legend_handles(stimulus_names, stimulus_colors), frameon=False, loc="upper left", borderaxespad=0.0)
full_raster_axis.set_xlim(recording_limits)
full_raster_axis.set_ylabel("Neuron (sorted rank)" if SORT_BY_CORRELATION else "Neuron")
full_raster_axis.set_title(f"{FISH_ID} merged dF/F")
full_mean_axis.set_xlabel("Time (s)")
full_mean_axis.set_ylabel("Mean dF/F")
plt.show()

## Cell 07 — Average responses by stimulus
Average the repetitions of each stimulus and concatenate those averages horizontally.

In [ ]:
aligned = build_trial_aligned_traces(
    dfof=dfof,
    stimulus_trace=stimuli["stimulus_trace"],
    stimulus_id_map=stimuli["stimulus_id_map"],
    stimulus_durations=stimuli["stimulus_durations"],
    imaging_fps=IMAGING_FPS,
    pre_stimulus_sec=PRE_STIMULUS_SEC,
    post_stimulus_margin_sec=POST_STIMULUS_MARGIN_SEC,
)
print(aligned["trial_counts"])
concatenated_responses = concatenate_stimulus_mean_rasters(
    mean_rasters=aligned["mean_rasters"],
    stimulus_durations=stimuli["stimulus_durations"],
    pre_frames=aligned["pre_frames"],
    imaging_fps=IMAGING_FPS,
    stimulus_order=STIMULUS_ORDER,
)
concatenated_raster = concatenated_responses["raster"]
stimulus_segments = concatenated_responses["segments"]
stimulus_neuron_order = (
    correlation_sort_order(concatenated_raster.T)
    if SORT_BY_CORRELATION
    else np.arange(concatenated_raster.shape[0])
)
# Raster row i shows the neuron described in stimulus_sorted_roi_mapping row i
stimulus_sorted_roi_mapping = roi_mapping.iloc[stimulus_neuron_order].reset_index(drop=True)

## Cell 08 — Plot concatenated stimulus averages
Raster of the concatenated averages with segment boundaries (dashed), presentation onsets (dotted) and movement onsets (coloured).

In [ ]:
segment_time_sec = np.arange(concatenated_raster.shape[1]) / IMAGING_FPS
total_duration_sec = concatenated_raster.shape[1] / IMAGING_FPS
segment_limits = (0.0, total_duration_sec)
segment_names = stimulus_segments["stimulus_name"].tolist()
segment_boundaries_sec = [*(stimulus_segments["start_frame"] / IMAGING_FPS), total_duration_sec]
presentation_onsets_sec = stimulus_segments["presentation_onset_frame"] / IMAGING_FPS
segment_movement_onsets_sec = stimulus_segments["movement_onset_frame"] / IMAGING_FPS
stimulus_raster = concatenated_raster[stimulus_neuron_order, :]
stimulus_vmax = RASTER_VMAX if RASTER_VMAX is not None else robust_color_limit([concatenated_raster], RASTER_PERCENTILE, RASTER_VMIN)

stimulus_raster_figure = plt.figure(figsize=ACTIVITY_FIGURE_SIZE, constrained_layout=True)
stimulus_grid = stimulus_raster_figure.add_gridspec(2, 3, width_ratios=ACTIVITY_WIDTH_RATIOS, height_ratios=ACTIVITY_HEIGHT_RATIOS)
stimulus_raster_axis = stimulus_raster_figure.add_subplot(stimulus_grid[0, 0])
stimulus_mean_axis = stimulus_raster_figure.add_subplot(stimulus_grid[1, 0], sharex=stimulus_raster_axis)
stimulus_colorbar_axis = stimulus_raster_figure.add_subplot(stimulus_grid[0, 1])
stimulus_legend_axis = stimulus_raster_figure.add_subplot(stimulus_grid[0, 2])

draw_raster(stimulus_raster_axis, stimulus_raster, *segment_limits, RASTER_CMAP, RASTER_VMIN, stimulus_vmax)
draw_population_mean(stimulus_mean_axis, stimulus_raster, segment_time_sec, color="black")
for axis in (stimulus_raster_axis, stimulus_mean_axis):
    draw_vertical_lines(axis, segment_boundaries_sec, "0.65", segment_limits, linestyle="--", linewidth=0.8)
    draw_vertical_lines(axis, presentation_onsets_sec, "0.35", segment_limits, linestyle=":", linewidth=0.9)
    draw_stimulus_markers(axis, segment_movement_onsets_sec, segment_names, stimulus_colors, segment_limits, linewidth=1.4)
    axis.spines[["top", "right"]].set_visible(False)

segment_label_axis = stimulus_raster_axis.secondary_xaxis("top")
segment_label_axis.set_xticks(stimulus_segments["center_frame"] / IMAGING_FPS, segment_names, rotation=45, ha="left")
segment_label_axis.tick_params(length=0, pad=4)
stimulus_raster_figure.colorbar(stimulus_raster_axis.images[-1], cax=stimulus_colorbar_axis, label="Mean dF/F")
stimulus_legend_axis.axis("off")
stimulus_legend_axis.legend(handles=stimulus_legend_handles(segment_names, stimulus_colors), title="Movement onset", frameon=False, loc="upper left", borderaxespad=0.0)
stimulus_raster_axis.set_xlim(segment_limits)
stimulus_raster_axis.set_ylabel("Neuron (sorted rank)" if SORT_BY_CORRELATION else "Neuron")
stimulus_raster_axis.set_title(f"{FISH_ID} average responses by stimulus")
stimulus_mean_axis.set_xlabel("Concatenated stimulus time (s)")
stimulus_mean_axis.set_ylabel("Mean dF/F")
plt.show()

## Cell 09 — Compute response metrics
Neuron-level AUC and maximum from onset to the end of each stimulus's window (stimulus + margin), summarised as population mean ± SEM.

In [ ]:
response_metrics = compute_response_metrics(
    mean_rasters=aligned["mean_rasters"],
    pre_frames=aligned["pre_frames"],
    imaging_fps=IMAGING_FPS,
)
auc_summary = summarize_metric(response_metrics["auc"])
maximum_summary = summarize_metric(response_metrics["maximum_amplitude"])

## Cell 10 — Plot response metrics
One bar per stimulus, coloured like the rasters.

In [ ]:
metric_panels = [
    (auc_summary, "AUC (dF/F × s)", "Response AUC"),
    (maximum_summary, "Maximum dF/F", "Maximum response amplitude"),
]
metric_figures = []
for metric_summary, metric_label, metric_title in metric_panels:
    bar_positions = np.arange(len(metric_summary))
    bar_colors = [stimulus_colors[stimulus_name] for stimulus_name in metric_summary["stimulus"]]
    metric_figure, metric_axis = plt.subplots(figsize=METRIC_FIGURE_SIZE)
    metric_axis.bar(bar_positions, metric_summary["mean"], yerr=metric_summary["sem"], color=bar_colors, edgecolor="black", linewidth=0.6, capsize=3)
    metric_axis.axhline(0.0, color="black", linewidth=0.7)
    metric_axis.set_xticks(bar_positions, metric_summary["stimulus"], rotation=45, ha="right")
    metric_axis.set_ylabel(metric_label)
    metric_axis.set_title(metric_title)
    metric_axis.spines[["top", "right"]].set_visible(False)
    metric_figure.tight_layout()
    metric_figures.append(metric_figure)
    plt.show()
auc_figure, maximum_figure = metric_figures

## Cell 11 — Save the figures
Save PNG copies only when `SAVE_PLOTS` is enabled; existing figures are never overwritten.

In [16]:
saved_paths = []
if SAVE_PLOTS:
    saved_paths.append(save_figure_safely(full_figure, PLOTS_DIR, f"{FISH_ID}_full_activity.png", PLOT_DPI))
    saved_paths.append(save_figure_safely(stimulus_raster_figure, PLOTS_DIR, f"{FISH_ID}_stimulus_rasters.png", PLOT_DPI))
    saved_paths.append(save_figure_safely(auc_figure, PLOTS_DIR, f"{FISH_ID}_stimulus_auc.png", PLOT_DPI))
    saved_paths.append(save_figure_safely(maximum_figure, PLOTS_DIR, f"{FISH_ID}_stimulus_max_amplitude.png", PLOT_DPI))
print("Saved figures:", saved_paths if saved_paths else "saving disabled")

Saved figures: [WindowsPath('C:/Users/suribear/OneDrive - Université de Lausanne/Lab/Data/2p/L588_f01/04_plots/basic_calcium_analysis/L588_f01_full_activity.png'), WindowsPath('C:/Users/suribear/OneDrive - Université de Lausanne/Lab/Data/2p/L588_f01/04_plots/basic_calcium_analysis/L588_f01_stimulus_rasters.png'), WindowsPath('C:/Users/suribear/OneDrive - Université de Lausanne/Lab/Data/2p/L588_f01/04_plots/basic_calcium_analysis/L588_f01_stimulus_auc.png'), WindowsPath('C:/Users/suribear/OneDrive - Université de Lausanne/Lab/Data/2p/L588_f01/04_plots/basic_calcium_analysis/L588_f01_stimulus_max_amplitude.png')]
